In [1]:
import zipfile, glob, sqlite3
import pandas as pd
from pathlib import Path

# ── paths ──────────────────────────────────────────────
RAW = Path("../data/raw")
DB  = Path("../data/baywheels.db")
DB.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB)
print(f"Database: {DB}")

# ── column normalizer ───────────────────────────────────
# Bay Wheels changed column names when Lyft took over
def normalize(df):
    rename = {
        "Start Time":               "started_at",
        "Stop Time":                "ended_at",
        "Start Station ID":         "start_station_id",
        "Start Station Name":       "start_station_name",
        "Start Station Latitude":   "start_lat",
        "Start Station Longitude":  "start_lng",
        "End Station ID":           "end_station_id",
        "End Station Name":         "end_station_name",
        "End Station Latitude":     "end_lat",
        "End Station Longitude":    "end_lng",
        "User Type":                "member_casual",
        "Trip Duration":            "duration_sec",
        "Bike ID":                  "ride_id",
    }
    df = df.rename(columns=rename)

    keep = ["ride_id","rideable_type","started_at","ended_at",
            "start_station_id","start_station_name",
            "end_station_id","end_station_name",
            "start_lat","start_lng","end_lat","end_lng",
            "member_casual"]

    for col in keep:
        if col not in df.columns:
            df[col] = None

    return df[keep]

# ── load all zips ───────────────────────────────────────
zips = sorted(RAW.glob("*.zip"))
print(f"Loading {len(zips)} files...\n")

for i, zf in enumerate(zips, 1):
    print(f"[{i}/{len(zips)}] {zf.name}", end=" ... ")
    with zipfile.ZipFile(zf) as z:
        csv_name = z.namelist()[0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f, low_memory=False)
    df = normalize(df)
    df.to_sql("trips", conn, if_exists="append", index=False)
    print(f"{len(df):,} rows")

# ── build stations dimension ────────────────────────────
print("\nBuilding stations table...")
stations = pd.read_sql("""
    SELECT start_station_id  AS station_id,
           start_station_name AS station_name,
           AVG(start_lat)    AS lat,
           AVG(start_lng)    AS lng
    FROM trips
    WHERE start_station_id IS NOT NULL
    GROUP BY start_station_id, start_station_name
""", conn)
stations.to_sql("stations", conn, if_exists="replace", index=False)
print(f"{len(stations):,} stations")

# ── verify ──────────────────────────────────────────────
total = pd.read_sql("SELECT COUNT(*) AS n FROM trips", conn).iloc[0,0]
print(f"\nTotal trips loaded: {total:,}")
print("Tables:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist())

conn.close()
print("\nDone. baywheels.db is ready.")

Database: ../data/baywheels.db
Loading 59 files...

[1/59] 202001-baywheels-tripdata.csv.zip ... 295,854 rows
[2/59] 202002-baywheels-tripdata.csv.zip ... 432,354 rows
[3/59] 202003-baywheels-tripdata.csv.zip ... 176,799 rows
[4/59] 202005-baywheels-tripdata.csv.zip ... 138,522 rows
[5/59] 202006-baywheels-tripdata.csv.zip ... 158,410 rows
[6/59] 202007-baywheels-tripdata.csv.zip ... 155,248 rows
[7/59] 202008-baywheels-tripdata.csv.zip ... 152,690 rows
[8/59] 202009-baywheels-tripdata.csv.zip ... 144,348 rows
[9/59] 202010-baywheels-tripdata.csv.zip ... 167,541 rows
[10/59] 202011-baywheels-tripdata.csv.zip ... 133,020 rows
[11/59] 202012-baywheels-tripdata.csv.zip ... 106,422 rows
[12/59] 202101-baywheels-tripdata.csv.zip ... 102,363 rows
[13/59] 202102-baywheels-tripdata.csv.zip ... 111,073 rows
[14/59] 202103-baywheels-tripdata.csv.zip ... 131,960 rows
[15/59] 202104-baywheels-tripdata.csv.zip ... 146,002 rows
[16/59] 202105-baywheels-tripdata.csv.zip ... 169,642 rows
[17/59] 20210

In [2]:
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect("../data/baywheels.db")

weather = pd.read_csv("../data/weather_2018_2024.csv")
weather["DATE"] = pd.to_datetime(weather["DATE"]).dt.date.astype(str)
weather.to_sql("weather", conn, if_exists="replace", index=False)

print(f"Weather rows: {len(weather):,}")
print("Tables:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist())
conn.close()

Weather rows: 2,557
Tables: ['trips', 'stations', 'weather']
